In [ ]:
import os
from odf.opendocument import load
from odf import table, text
from math import radians, sin, cos, sqrt, atan2
import xarray as xr
import numpy as np
from seabird.cnv import fCNV
import res
from datetime import datetime, time

import pandas as pd
from datetime import datetime
from datetime import datetime, timedelta
import numpy as np

In [4]:
def dm_to_dd(deg, minutes, west=False, south=False):
    """Convert degrees/minutes to decimal degrees."""
    dd = float(deg) + float(minutes) / 60.0
    if west or south:
        dd *= -1
    return dd

def get_next_value(values, key):
    for i, v in enumerate(values):
        if isinstance(v, str) and v.strip().startswith(key):
            if i + 1 < len(values):
                return values[i + 1]
    return None


def haversine(lon1, lat1, lon2, lat2):
    R = 6371  # km
    dlon = radians(lon2 - lon1)
    dlat = radians(lat2 - lat1)

    a = sin(dlat/2)**2 + cos(radians(lat1))*cos(radians(lat2))*sin(dlon/2)**2
    return 2 * R * atan2(sqrt(a), sqrt(1 - a))


def assign_stations(casts, threshold_km=2):
    station_id = 1
    station_letters = {}

    prev_lon = None
    prev_lat = None
    letter_counter = {}

    for i, c in enumerate(casts):

        if i == 0:
            c["station_number"] = station_id
            letter_counter[station_id] = 0
            c["station_id"] = f"{station_id}a"
            prev_lon, prev_lat = c["lon_start"], c["lat_start"]
            continue

        dist = haversine(prev_lon, prev_lat, c["lon_start"], c["lat_start"])

        if dist < threshold_km:
            # same station
            c["station_number"] = station_id
        else:
            station_id += 1
            c["station_number"] = station_id
            letter_counter[station_id] = 0

        # station letter (a, b, c...)
        letter_counter[c["station_number"]] += 1
        letter = chr(96 + letter_counter[c["station_number"]])  # a,b,c...

        c["station_id"] = f"{c['station_number']}{letter}"

        prev_lon, prev_lat = c["lon_start"], c["lat_start"]

    return casts


def save_to_nc(casts, outfile="ctd_metadata.nc"):

    n = len(casts)

    ds = xr.Dataset(
        data_vars=dict(
            station_number=("cast", [c["station_number"] for c in casts]),
            cast_id=("cast", [c["cast_id"] for c in casts]),
            depth=("cast", [c["depth"] for c in casts]),

            lon_start=("cast", [c["lon_start"] for c in casts]),
            lat_start=("cast", [c["lat_start"] for c in casts]),
            lon_end=("cast", [c["lon_end"] for c in casts]),
            lat_end=("cast", [c["lat_end"] for c in casts]),
            # time_start = ("cast", [c["time_start"] for c in casts]),
            # time_end = ("cast", [c["time_end"] for c in casts]),
        ),
        coords=dict(
            cast=np.arange(n),
            station_id=("cast", [c["station_id"] for c in casts]),
            time_start=("cast", [c["time_start"] for c in casts]),
            time_end=("cast", [c["time_end"] for c in casts]),
        )
    )

    ds.to_netcdf(outfile)
    print(f"Saved: {outfile}")

def det_max_depth(pres):
    if pres < 600:
        return 300
    elif pres < 1500:
        return 1200
    else:
        return 5000
    


In [5]:
ctd_path = "/Volumes/science/MD252_WHIRLS/CTD/data/"

In [6]:
files = []
ctds = os.listdir(ctd_path)
for ctd in ctds:
    if ctd[-7:-4] == "prs":
        files.append(ctd)

In [7]:
files = sorted(files)

In [ ]:
casts = []
cast_counter = 0

for f in files:
    year = 2026
    curr_ctd = fCNV(ctd_path + f)
    pressure = curr_ctd['PRES']
    # ctd type
    depth = det_max_depth(np.max(pressure))
    if cast_counter == 0:
        depth = 300

    # start time
    timeJ = np.asarray(curr_ctd["timeJ"].filled(np.nan), dtype=float)

    times = np.array([
        datetime(year, 1, 1) + timedelta(days=t - 1)
        for t in timeJ
    ])

    lat = curr_ctd["LATITUDE"]
    lon = curr_ctd["LONGITUDE"]
    i_start = 0
    i_end = -1

    time_start = times[i_start]
    lon_start = lon[i_start]
    lat_start = lat[i_start]

    time_end = times[i_end]
    lon_end = lon[i_end]
    lat_end = lat[i_end]
    cast_counter += 1
    casts.append({
            "cast_id": cast_counter,
            "lon_start": lon_start,
            "lat_start": lat_start,
            "lon_end": lon_end,
            "lat_end": lat_end,
            "depth": depth,
            "time_start": time_start,
            "time_end": time_end,
        })

It was supposed to has 21 variables.
It was supposed to has 21 variables.
It was supposed to has 21 variables.
It was supposed to has 21 variables.
It was supposed to has 21 variables.
It was supposed to has 21 variables.
It was supposed to has 21 variables.
It was supposed to has 21 variables.
It was supposed to has 21 variables.
It was supposed to has 21 variables.
It was supposed to has 21 variables.
It was supposed to has 21 variables.
It was supposed to has 21 variables.
It was supposed to has 21 variables.
It was supposed to has 21 variables.
It was supposed to has 21 variables.
It was supposed to has 21 variables.
It was supposed to has 21 variables.
It was supposed to has 21 variables.
It was supposed to has 21 variables.
It was supposed to has 21 variables.
It was supposed to has 21 variables.
It was supposed to has 21 variables.
It was supposed to has 21 variables.
It was supposed to has 21 variables.
It was supposed to has 21 variables.
It was supposed to has 21 variables.
I

In [9]:
casts = assign_stations(casts, threshold_km=0.1)
save_to_nc(casts)

Saved: ctd_metadata.nc


In [11]:
import xarray as xr
path = "ctd_metadata.nc"
ds = xr.open_dataset(path)

In [12]:
ds.close()

In [13]:
ds

<xarray.Dataset> Size: 7kB
Dimensions:         (cast: 77)
Coordinates:
  * cast            (cast) int64 616B 0 1 2 3 4 5 6 7 ... 70 71 72 73 74 75 76
    station_id      (cast) <U3 924B ...
    time_start      (cast) datetime64[ns] 616B ...
    time_end        (cast) datetime64[ns] 616B ...
Data variables:
    station_number  (cast) int64 616B ...
    cast_id         (cast) int64 616B ...
    depth           (cast) int64 616B ...
    lon_start       (cast) float64 616B ...
    lat_start       (cast) float64 616B ...
    lon_end         (cast) float64 616B ...
    lat_end         (cast) float64 616B ...

In [14]:
ds['cast_id'].values

array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
       18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34,
       35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51,
       52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68,
       69, 70, 71, 72, 73, 74, 75, 76, 77])

In [15]:
ds['station_number'].values

array([ 1,  1,  2,  3,  3,  4,  5,  6,  6,  7,  8,  9, 10, 11, 12, 13, 14,
       15, 15, 16, 17, 18, 19, 20, 21, 21, 22, 23, 24, 25, 26, 27, 28, 29,
       30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 45,
       46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62,
       63, 64, 65, 66, 67, 68, 69, 70, 71])

In [21]:
ds['time_start'].values

array(['2026-06-29T20:41:37.104000000', '2026-06-29T22:02:59.136000000',
       '2026-06-30T10:31:27.436800000', '2026-07-01T17:13:39.561600000',
       '2026-07-01T18:32:04.300800000', '2026-07-02T05:35:49.516800000',
       '2026-07-02T06:46:06.960000000', '2026-07-02T20:48:55.324800000',
       '2026-07-02T21:49:36.048000000', '2026-07-03T07:06:03.600000000',
       '2026-07-04T00:52:22.540800000', '2026-07-04T02:03:03.139200000',
       '2026-07-04T05:10:00.278400000', '2026-07-04T06:38:19.795200000',
       '2026-07-04T09:36:38.880000000', '2026-07-04T11:11:10.953600000',
       '2026-07-04T14:14:52.915200000', '2026-07-04T16:07:20.841600000',
       '2026-07-04T17:05:55.334400000', '2026-07-05T07:25:01.142400000',
       '2026-07-05T08:56:53.203200000', '2026-07-05T11:36:18.633600000',
       '2026-07-05T13:12:03.369600000', '2026-07-05T15:57:50.601600000',
       '2026-07-05T17:34:29.424000000', '2026-07-05T18:28:33.484800000',
       '2026-07-05T21:18:17.798400000', '2026-07-05

In [22]:
ds['depth'].values

array([ 300, 1200, 5000,  300, 1200,  300, 1200,  300, 1200, 1200,  300,
       1200, 1200, 1200, 1200, 1200, 1200,  300, 1200, 1200, 1200, 1200,
       1200, 1200,  300, 1200, 1200, 1200, 1200, 1200, 1200,  300, 1200,
       1200, 1200, 1200, 1200, 1200, 1200, 1200,  300, 1200,  300, 1200,
       1200, 1200, 1200, 1200, 1200,  300, 1200, 1200, 1200, 1200, 1200,
       1200,  300, 1200, 1200, 1200, 1200, 1200, 1200,  300, 1200, 1200,
       1200, 1200, 1200, 5000, 1200,  300, 1200, 1200,  300, 1200])